# Generate restored SPFC on GenEval 100 (Kaggle)

Runs Sparse Primitive Flow Composition with authored condition weights, consensus gating, and target-consistency gating for the checked-in 100-prompt GenEval subset.

Outputs are written under `/kaggle/working/geneval_seed13/runs/geneval/spfc` and persist as this notebook's Kaggle output dataset.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import shlex
import subprocess
import sys
import time

os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("DIFFUSERS_VERBOSITY", "error")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTHONUNBUFFERED", "1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")


SEED = 13
RUN_SLUG = "geneval_seed13"
LOCAL_REPO_DIR = Path(os.environ.get("AIM_FLOW_REPO_DIR", "/media/fezan/ASi/DVLM/steering/aim-flow")).expanduser()
IS_KAGGLE = Path("/kaggle").exists()
WORK_ROOT = Path("/kaggle/working") if IS_KAGGLE else LOCAL_REPO_DIR
OUTPUT_ROOT = WORK_ROOT / RUN_SLUG
RUN_ROOT = OUTPUT_ROOT / "runs"
MANIFEST_REL = Path("configs/geneval_100_seed13.json")
DECOMP_REL = Path("configs/geneval_100_seed13_spfc.json")
RECTIFIED_REPO_DIR = WORK_ROOT / "Rectified-CFGpp"
INSTALL_DEPS = True

# If your repo is attached with a different Kaggle dataset slug, this auto-discovers it under /kaggle/input.
def has_aim_flow_repo(path: Path) -> bool:
    return (path / "src" / "aim_flow").exists() and (path / "scripts" / "bench_generate.py").exists()


def find_aim_flow_repo() -> Path | None:
    cwd = Path.cwd()
    local_candidates = [cwd, *cwd.parents, LOCAL_REPO_DIR]
    for candidate in local_candidates:
        if has_aim_flow_repo(candidate):
            return candidate
    dest = WORK_ROOT / "aim-flow"
    if has_aim_flow_repo(dest):
        return dest
    input_root = Path("/kaggle/input")
    candidates = [input_root / "aim-flow", input_root / "aim-flow" / "aim-flow"]
    if input_root.exists():
        for child in sorted(input_root.glob("*")):
            candidates.extend([child, child / "aim-flow"])
    for candidate in candidates:
        if has_aim_flow_repo(candidate):
            return candidate
    return None


def ensure_working_repo() -> Path:
    source = find_aim_flow_repo()
    if source is None:
        raise FileNotFoundError(
            "Could not find aim-flow. Attach the repo as a Kaggle dataset, clone it into /kaggle/working, "
            "or run this notebook from the repo root."
        )
    dest = WORK_ROOT / "aim-flow" if IS_KAGGLE else source
    if source.resolve() != dest.resolve():
        shutil.copytree(source, dest, dirs_exist_ok=True)
        return dest
    return source


def run_args(args: list[str], cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess:
    print("$", shlex.join([str(arg) for arg in args]))
    started = time.time()
    result = subprocess.run([str(arg) for arg in args], cwd=str(cwd) if cwd else None, text=True)
    print(f"elapsed: {(time.time() - started) / 60:.2f} min")
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result


REPO_DIR = ensure_working_repo()
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))
print("repo:", REPO_DIR)
print("outputs:", OUTPUT_ROOT)

if INSTALL_DEPS:
    run_args([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], cwd=REPO_DIR)
    run_args([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], cwd=REPO_DIR)

In [ ]:
METHOD = "spfc"
METHOD_TITLE = "Restored SPFC"
GUIDANCE_SCALE = 4.5
print(f"Generating {METHOD_TITLE} with method={METHOD}, seed={SEED}, guidance_scale={GUIDANCE_SCALE}")

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        if not os.environ.get("HF_TOKEN"):
            try:
                value = secrets.get_secret(secret_name)
                if value:
                    os.environ["HF_TOKEN"] = value
            except Exception:
                pass
except Exception:
    pass

if os.environ.get("HF_TOKEN"):
    print("HF_TOKEN is available.")
else:
    print("HF_TOKEN is not set. Add it as a Kaggle secret before loading the gated SD3 model.")

run_args(["nvidia-smi"], check=False)

In [ ]:
manifest_path = REPO_DIR / MANIFEST_REL
if not manifest_path.exists():
    raise FileNotFoundError(manifest_path)

with manifest_path.open("r", encoding="utf-8") as f:
    manifest = json.load(f)

sample_count = len(manifest["samples"])
print("manifest:", manifest_path)
print("benchmark:", manifest["benchmark"])
print("samples:", sample_count)
assert sample_count == 100, f"Expected 100 GenEval samples, found {sample_count}."

if METHOD == "spfc":
    decomp_path = REPO_DIR / DECOMP_REL
    if not decomp_path.exists():
        raise FileNotFoundError(decomp_path)
    with decomp_path.open("r", encoding="utf-8") as f:
        decompositions = json.load(f)
    print("decompositions:", decomp_path)
    assert len(decompositions["items"]) == 100, "SPFC decomposition file must cover all 100 samples."

    from aim_flow.eval_bench.generation import load_bench_config

    config = load_bench_config(seed=SEED, guidance_scale=GUIDANCE_SCALE)
    primitive_flow = config.primitive_flow
    assert primitive_flow.uniform_condition_weights is False
    assert primitive_flow.use_consensus_gating is True
    assert primitive_flow.use_target_consistency_gating is True
    assert primitive_flow.source_weight == 0.7
    assert primitive_flow.target_weight == 1.2
    print("vfa: r_i = w_i * g_i^cons * g_i^tgt")
    print("weights: source=0.7, primitives=authored decomposition weights, target=1.2")
    print("gates: consensus=true, target_consistency=true")

In [ ]:
manifest_path = REPO_DIR / MANIFEST_REL
cmd = [
    sys.executable,
    "scripts/bench_generate.py",
    "--manifest",
    manifest_path,
    "--run-root",
    RUN_ROOT,
    "--methods",
    METHOD,
    "--seed",
    str(SEED),
    "--guidance-scale",
    str(GUIDANCE_SCALE),
    "--skip-existing",
]

if METHOD == "spfc":
    cmd.extend([
        "--decompositions",
        REPO_DIR / DECOMP_REL,
    ])
if METHOD == "rectified_cfgpp":
    cmd.extend(["--rectified-repo-dir", RECTIFIED_REPO_DIR])

run_args(cmd, cwd=REPO_DIR)

In [ ]:
method_dir = RUN_ROOT / "geneval" / METHOD
index_path = method_dir / "index.json"
images = sorted(method_dir.glob("*.png"))
metadata = sorted(method_dir.glob("*.json"))
metadata = [path for path in metadata if path.name != "index.json"]

print("method_dir:", method_dir)
print("index:", index_path)
print("images:", len(images))
print("metadata files:", len(metadata))
assert index_path.exists(), f"Missing generation index: {index_path}"
assert len(images) == 100, f"Expected 100 images for {METHOD}, found {len(images)}."

readme = OUTPUT_ROOT / f"README_{METHOD}.txt"
readme.write_text(
    "GenEval 100 generated images\n"
    f"method: {METHOD}\n"
    "variant: full\n"
    "vfa: r_i = w_i * g_i^cons * g_i^tgt\n"
    "uniform_condition_weights: false\n"
    "use_consensus_gating: true\n"
    "use_target_consistency_gating: true\n"
    "source_weight: 0.7\n"
    "target_weight: 1.2\n"
    "primitive_weights: authored decomposition weights\n"
    f"seed: {SEED}\n"
    f"guidance_scale: {GUIDANCE_SCALE}\n"
    f"image_dir: {method_dir}\n",
    encoding="utf-8",
)
archive_path = Path(shutil.make_archive(str(OUTPUT_ROOT), "zip", root_dir=OUTPUT_ROOT.parent, base_dir=OUTPUT_ROOT.name))
print("output root:", OUTPUT_ROOT)
print("zip archive:", archive_path)